# **Homework Assignment QT2: Gene Finding**

**Name:** Vedasravas Dasari

**M ID:** M16446787

**Colab Link:** https://colab.research.google.com/drive/1KoS5J_WQNzGgyekpXFG0l_EjqDDuZ_ew?usp=sharing

**Submission Date:** 02/20/2025

---

**Course:** CS7099-Introduction to Bioinformatics

**Instructor:** Jarek Meller

---

Step 1: Prepare input and the environment

In [2]:
import sys
import os
from google.colab import drive
drive.mount('/content/drive')
genome_file_name = "mt_human.fasta"
!wget https://ftp.ensembl.org/pub/current_fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.chromosome.MT.fa.gz
!gunzip -f Homo_sapiens.GRCh38.dna.chromosome.MT.fa.gz
!cp Homo_sapiens.GRCh38.dna.chromosome.MT.fa $genome_file_name
!ls -al
!head -n 10 $genome_file_name


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--2025-02-20 23:54:49--  https://ftp.ensembl.org/pub/current_fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.chromosome.MT.fa.gz
Resolving ftp.ensembl.org (ftp.ensembl.org)... 193.62.193.169
Connecting to ftp.ensembl.org (ftp.ensembl.org)|193.62.193.169|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5400 (5.3K) [application/x-gzip]
Saving to: ‘Homo_sapiens.GRCh38.dna.chromosome.MT.fa.gz’

Homo_sapiens.GRCh38 100%[===================>]   5.27K  --.-KB/s    in 0s      

2025-02-20 23:54:50 (1.28 GB/s) - ‘Homo_sapiens.GRCh38.dna.chromosome.MT.fa.gz’ saved [5400/5400]

total 60
drwxr-xr-x 1 root root  4096 Feb 20 23:54 .
drwxr-xr-x 1 root root  4096 Feb 20 23:51 ..
drwxr-xr-x 4 root root  4096 Feb 19 18:24 .config
drwx------ 6 root root  4096 Feb 20 23:53 drive
-rw-r--r-- 1 root root 16900 Aug 15  2024 Homo_sapiens.GRCh38.dna.chromosome.

Step 2: Read the genome

In [3]:
#
# READ THE GENOME
#
print("Reading the genome from ",genome_file_name," \n")
!head -10 mt_human.fasta
# Open and read the genome file, skip the first line starting with >
# Concatenate all other lines as your genome
FILE = open(genome_file_name)
FILE.readline()
genome = ''
for genome_line in FILE:
	genome_line = genome_line.rstrip()
	genome = genome + genome_line
#genome.strip()
#print(genome)

Reading the genome from  mt_human.fasta  

>MT dna:chromosome chromosome:GRCh38:MT:1:16569:1 REF
GATCACAGGTCTATCACCCTATTAACCACTCACGGGAGCTCTCCATGCATTTGGTATTTT
CGTCTGGGGGGTATGCACGCGATAGCATTGCGAGACGCTGGAGCCGGAGCACCCTATGTC
GCAGTATCTGTCTTTGATTCCTGCCTCATCCTATTATTTATCGCACCTACGTTCAATATT
ACAGGCGAACATACTTACTAAAGTGTGTTAATTAATTAATGCTTGTAGGACATAATAATA
ACAATTGAATGTCTGCACAGCCACTTTCCACACAGACATCATAACAAAAAATTTCCACCA
AACCCCCCCTCCCCCGCTTCTGGCCACAGCACTTAAACACATCTCTGCCAAACCCCAAAA
ACAAAGAACCCTAACACCAGCCTAACCAGATTTCAAATTTTATCTTTTGGCGGTATGCAC
TTTTAACAGTCACCCCCCAACTAACACATTATTTTCCCCTCCCACTCCCATACTACTAAT
CTCATCAATACAACCCCCGCCCATCCTACCCAGCACACACACACCGCTGCTAACCCCATA


Step 2a: This where you can also define the reverse complement in order to find genes on the 'negative' strand.

Step 3: Define the genetic code - modify here as needed to use the MT code

In [ ]:
starts_standard_genetic_code = ["ATG"]
starts_mito = ["ATG","ATA"]
starts = starts_mito
print("Start codons:")
for codons in starts:
  print(codons)
stops_standard_genetic_code = ["TAG","TGA","TAA"]
stops_mito = ["TAG","TAA","AGA","AGG"]
stops = stops_mito
print("Stop codons")
for codons in stops:
  print(codons)


Study carefully the way stop codons are used by the mitochondria. Does that inform a better strategy that may limit false positive rates while still detecting ORF containing genes? Discuss here and modify the code accordingly.

A propos, what about that one pesky gene that starts with ATT ?

Well, in some vertebrate species, some mtDNA genes start with codon sequences ATT but these are still translated as Methionine rather than Isoleucine. speculate as to what is the mechanism for this non-standard translation? How would you detect those without increasing the false positive rate?

You can find more about the mitochondrial genetic code here: [Wiki on mitogenome](https://en.wikipedia.org/wiki/Vertebrate_mitochondrial_code)

Step 4: Search for ORFs

Step 4a: Define a function to find ORFs in a reading frame

In [ ]:
def find_orfs(genome,reading_frame,start_codon,stops,len_threshold,f_out):
  len_codon = 3
  len_genome = len(genome)
  n_stops = len(stops)
  #print("Using ",n_stops," stops:")
  #for codons in stops:
  #  print(codons)
  len_orf = 0
  start_orf = 0
  orf_started= 0
  i = reading_frame
  while(i<len_genome-len_codon+1):
    kmer = genome[i:i+len_codon]
    if ((kmer == start_codon) & (orf_started == 0)):
      start_orf = i
      len_orf = 1
      orf_started = 1
			#print(" ORF started at position ",i," with start= ",kmer)
    j = 0
    if(orf_started == 1):
      while(j < n_stops):
        stop_codon = stops[j]
        if (kmer == stop_codon):
          stop_orf = i
          len_orf = i - start_orf
          orf_started = 0
					#print(" stop found at position ",i,stop_codon," \n")
          if (len_orf > len_threshold):
            print(" ORF found at ",start_orf+1,stop_orf+3," \n",file = f_out)
            print(" ORF found at ",start_orf+1,stop_orf+3," \n")
            orf = genome[start_orf:stop_orf+3]
            print(orf,"\n")
						#print("stop found= ",stop_codon," \n")
        j = j + 1
    i = i + 3
  return

Step 4b: Loop over reading frames with some length threshold

In [ ]:
#
# Change the minumum length of ORFs to be reported to filter out false positives
#
len_threshold = 900
#
# Save the begin - end positions of predicted ORFs
#
output_file = "predicted_genes.txt"
f_out = open(output_file, "w")
#
# Find ORFs with each of the start codons considered
#
for start_codon in starts:
  #start_codon = starts[1]
  print("Using start_codon= ",start_codon,"\n")
  reading_frame = 0
  while(reading_frame<3):
    print("\n Reading frame ",reading_frame + 1," \n")
    find_orfs(genome,reading_frame,start_codon,stops,len_threshold,f_out)
    reading_frame = reading_frame + 1

# FIX ME: note that only the postive strand is considered
print("\n Done searching for ORFs \n")

# Collect your results
f_out.close()
!head -100 $output_file
!cp $output_file /content/drive/MyDrive/intro2bioinfo/.


Compare the results with the ground truth from BioMart (you may want to write some code to do this automatically). Experiment with the threshold for ORF length to generate predictions with acceptable false negative and false positive rates. Discuss potential discrepancies between ORF vs. actual gene start - end coordinates. Why are those observed?